# 0609ver_3 進度報告整理：原因對策 Service、API、Ontology / Knowledge 拆分

本 Notebook 用於說明 `0609ver_3` 中，針對下列三個方向所完成的整理內容：

1. 原因對策做成 service  
2. API 獨立整理出來  
3. knowledge / ontology 分開，不全部放在同一個 TTL 裡  

本版另外補強：

```text
DataPreprocess 端門檻參考值
sensor_threshold reference
EventRule 與 threshold reference 的關聯
OntoGraf 展示用關聯路徑
Ontology 各模組階層與關聯檢查
```

目前整體概念不是把 rule 拿掉，而是將功能拆清楚：

```text
EventRuleService / rules：負責判斷是否異常、是否觸發 alert_event
TroubleshootingService：負責在事件發生後，查可能原因與候選對策
Ontology：負責描述 sensor、threshold、rule、event、cause、response 之間的語意關係
Dashboard：負責把 alert、原因、對策顯示給使用者
```

整體流程如下：

```text
DataPreprocess
→ sensor_1hz / batch_summary / sensor_threshold reference
→ EventRuleService 參考 threshold 判斷異常
→ alert_event
→ TroubleshootingService 查原因對策
→ Dashboard 顯示可能原因與候選處置
```


## 1. 原因對策做成 Service

### 目前做到的內容

本版將「原因對策」整理成 `TroubleshootingService`，也就是原因對策查詢服務。

它的定位是：

```text
不是直接判斷異常
不是直接觸發 alert
而是在已經知道 issue_type / alert_event 後，查詢可能原因與候選處置
```

例如：

```text
FilterPressureHigh
→ 可能原因：濾網堵塞或材料雜質
→ 影響設備：FilterMesh
→ 觀測訊號：filter_diff_pressure_bar
→ 候選處置：清潔或更換濾網
```

目前這些原因對策先標成候選，不直接當成正式 rule。


### 1-1. 原因對策 Service 相關連檔案與目的

#### `webservices/troubleshooting_service_contract/troubleshooting_service_contract.py`

目的 / 主旨：

```text
定義 TroubleshootingService 未來應該提供哪些 function。
這是 service contract，不是正式後端實作。
```

主要 function 概念：

```text
get_troubleshooting_matrix()
→ 查整張原因對策矩陣

get_issue_recommendations(issue_type, station)
→ 根據某個異常類型與站別，查可能原因與候選對策
```

---

#### `knowledge/troubleshooting_matrix_candidate.csv`

目的 / 主旨：

```text
存放原因對策候選資料。
這份是 TroubleshootingService 未來要查的 knowledge source。
```

主要欄位：

```text
issue_type
fault_symptom
possible_cause
affected_asset
observed_signal
candidate_response
status
```

目前 `status` 先標成：

```text
candidate_pending_confirmation
```

代表這些是候選知識，尚未當作正式規則。

---

#### `schema/troubleshooting_matrix.schema.json`

目的 / 主旨：

```text
定義 GET /troubleshooting/issues 回傳整張原因對策矩陣時的資料格式。
```

---

#### `schema/troubleshooting_issue.schema.json`

目的 / 主旨：

```text
定義 GET /troubleshooting/issues/{issue_type} 回傳單一異常原因對策時的資料格式。
```

---

#### `templates/troubleshooting_matrix.template.json`

目的 / 主旨：

```text
提供整張原因對策矩陣的回傳格式範例。
這是 template，不是正式資料。
```

---

#### `templates/troubleshooting_issue.template.json`

目的 / 主旨：

```text
提供單一 issue_type 原因對策查詢的回傳格式範例。
這是 template，不是正式資料。
```


## 2. API 獨立整理出來

### 目前做到的內容

本版將 API 獨立整理在 contract 檔案中，讓前端、service、schema、DB table 之間可以對應。

原因對策相關 API 主要有兩條：

```text
GET /troubleshooting/issues
GET /troubleshooting/issues/{issue_type}
```

另外 rule / alert 相關 API 也保留：

```text
POST /rules/evaluate
GET /alerts/unacknowledged
POST /alerts/{event_id}/ack
```

所以目前不是用原因對策 service 取代 rule，而是將「判斷異常」與「查原因對策」分開。


### 2-1. API 說明

#### `GET /troubleshooting/issues`

用途：

```text
查詢整張原因對策矩陣。
可以依 asset_type 或 issue_type 篩選。
```

例子：

```text
GET /troubleshooting/issues?asset_type=FilterMesh
```

意思是：

```text
查詢 FilterMesh 相關的異常原因與候選處置。
```

---

#### `GET /troubleshooting/issues/{issue_type}`

用途：

```text
查詢單一異常類型的可能原因與候選處置。
```

例子：

```text
GET /troubleshooting/issues/FilterPressureHigh?station=Station_1
```

意思是：

```text
查詢 Station_1 發生濾網壓差異常時，可能原因與候選處置是什麼。
```

---

#### `POST /rules/evaluate`

用途：

```text
給 EventRuleService 使用。
根據 sensor data、sensor_threshold reference、rule condition 判斷是否產生 alert_event。
```

---

#### `GET /alerts/unacknowledged`

用途：

```text
查詢尚未確認的 alert_event。
Dashboard 可用這個 API 顯示目前需要處理的事件。
```


### 2-2. API 相關連檔案與目的

#### `docs/contracts/api_routes_0609.csv`

目的 / 主旨：

```text
整理全部 API 路由。
包含 API path、method、service function、service class、input params、response schema、讀取資料表、寫入資料表、狀態與備註。
```

重點 API：

```text
GET /troubleshooting/issues
GET /troubleshooting/issues/{issue_type}
POST /rules/evaluate
GET /alerts/unacknowledged
POST /alerts/{event_id}/ack
GET /time-series
GET /time-series/features
GET /batches
GET /dashboard/manager-summary
```

---

#### `docs/contracts/service_io_contract_0609.csv`

目的 / 主旨：

```text
整理 service function 的 input / output 對應。
用來補充 API routes，說明每個 service function 要吃什麼資料、回傳什麼資料。
```

---

#### `schema/troubleshooting_matrix.schema.json`

目的 / 主旨：

```text
對應 GET /troubleshooting/issues 的 response schema。
```

---

#### `schema/troubleshooting_issue.schema.json`

目的 / 主旨：

```text
對應 GET /troubleshooting/issues/{issue_type} 的 response schema。
```

---

#### `templates/troubleshooting_matrix.template.json`

目的 / 主旨：

```text
示範 GET /troubleshooting/issues 回傳資料長什麼樣子。
```

---

#### `templates/troubleshooting_issue.template.json`

目的 / 主旨：

```text
示範 GET /troubleshooting/issues/{issue_type} 回傳資料長什麼樣子。
```

---

#### `webservices/troubleshooting_service_contract/troubleshooting_service_contract.py`

目的 / 主旨：

```text
對應 API 後方的 TroubleshootingService function。
說明 API 呼叫後，後端 service 應該如何處理原因對策查詢。
```

---

#### `webservices/event_rule_service_contract/event_rule_service_contract.py`

目的 / 主旨：

```text
對應 POST /rules/evaluate。
說明 EventRuleService 未來如何根據 sensor data、threshold reference、rule condition 判斷是否產生 alert_event。
```


## 3. knowledge / ontology 分開，不全部放在同一個 TTL 裡

### 目前做到的內容

本版沒有把所有 ontology 都塞在同一個 TTL，而是依功能拆成模組。

目前 ontology 主要劃分為：

```text
ontology/
├─ SprayLine_ontology_index.ttl
├─ core/
├─ database/
├─ service/
├─ dashboard/
├─ event_rule/
├─ troubleshooting/
└─ knowledge/
```

`0609ver_3` 另外在 knowledge ontology 裡補進 threshold reference：

```text
ontology/knowledge/SprayLine_threshold_reference_knowledge.ttl
```

用途是呈現 DataPreprocess 端提供的門檻參考值，並讓 EventRule 可以連到這些 reference。


### 3-1. Ontology 相關連檔案與目的

#### `ontology/SprayLine_ontology_index.ttl`

目的 / 主旨：

```text
ontology 的總索引檔。
用來說明整體 ontology 模組如何被拆分與連結。
```

---

### core

#### `ontology/core/SprayLine_core_ontology.ttl`

目的 / 主旨：

```text
放最基礎的產線、站點、設備、感測訊號概念。
```

包含概念：

```text
Station
PhysicalAsset
SensorSignal
RobotArm
FilterMesh
Nozzle
AirCompressor
SprayWidth
QualityModule
Oven
```

---

### database

#### `ontology/database/SprayLine_database_ontology.ttl`

目的 / 主旨：

```text
描述 DB Schema v3 的資料表與 view。
讓 ontology 可以對應資料庫中的資料來源。
```

包含：

```text
station_config
sensor_threshold
batch_run
sensor_1hz
batch_summary
pdm_degradation_log
alert_event
```

---

### service

#### `ontology/service/SprayLine_service_ontology.ttl`

目的 / 主旨：

```text
描述 Service、API endpoint、response schema 之間的關係。
```

包含：

```text
TimeSeriesService
BatchService
EventRuleService
TroubleshootingService
DashboardService
RESTEndpoint
ResponseSchema
```

原因對策的 `TroubleshootingService` 與 rule 判斷的 `EventRuleService` 都是在這一層描述。

---

### dashboard

#### `ontology/dashboard/SprayLine_dashboard_ontology.ttl`

目的 / 主旨：

```text
描述 Dashboard / UI 會顯示哪些資料區塊。
```

包含：

```text
ManagerDashboard
EngineerDashboard
AlertPanel
BatchRiskTable
SensorTrendChart
TroubleshootingPanel
```

其中：

```text
TroubleshootingPanel
```

是未來顯示原因與候選對策的 UI 區塊。

---

### event_rule

#### `ontology/event_rule/SprayLine_event_rule_ontology.ttl`

目的 / 主旨：

```text
描述 rule / event trigger 的概念。
這一層負責「判斷是否異常」與「是否產生 alert_event」。
```

包含：

```text
EventRule
TriggerRule
RuleCondition
RuleAction
AlertEvent
RiskLevelRule
SensorThresholdReference
```

`0609ver_3` 重要補強：

```text
EventRule
→ usesThresholdReference
→ SensorThresholdReference
```

也就是 EventRule 不只是空概念，而是可以連到 DataPreprocess threshold reference。

---

### troubleshooting

#### `ontology/troubleshooting/SprayLine_troubleshooting_ontology.ttl`

目的 / 主旨：

```text
描述原因對策的概念模型。
這裡定義原因對策會用到的類別。
```

包含：

```text
FaultSymptom
FaultCause
AffectedAsset
ObservedSignal
CandidateResponse
RecommendedResponse
```

這層是概念層，不是具體資料。

---

### knowledge

#### `ontology/knowledge/SprayLine_troubleshooting_knowledge.ttl`

目的 / 主旨：

```text
存放具體候選原因對策知識。
例如某個 issue_type 可能對應什麼原因、影響哪個設備、建議什麼處置。
```

它對應到：

```text
knowledge/troubleshooting_matrix_candidate.csv
```

也就是：

```text
CSV 是表格版候選知識
TTL 是 ontology graph 版候選知識
```

---

#### `ontology/knowledge/SprayLine_threshold_reference_knowledge.ttl`

目的 / 主旨：

```text
存放 DataPreprocess 端門檻參考值的 ontology graph 版本。
用來讓 SensorSignal、SensorThresholdReference、EventRule 之間建立可展示的關聯。
```

對應到：

```text
docs/contracts/data_preprocess_threshold_reference.csv
csv_templates/sensor_threshold_template.csv
```

---

#### `knowledge/troubleshooting_matrix_candidate.csv`

目的 / 主旨：

```text
存放原因對策候選知識的表格版本。
方便用 Excel 檢視，也方便後續 service 查詢。
```


## 4. DataPreprocess Threshold Reference 做了什麼

`0609ver_3` 已經將 DataPreprocess 端提供的門檻參考值整理進專案。

目前包含的感測訊號：

```text
filter_diff_pressure_bar
servo_torque_load_pct
spray_width_mm
film_thickness_um
paint_flow_ml_min
vibration_g
path_error_mm
gearbox_temperature_c
temperature_c
humidity_rh
air_pressure_bar
```

目前狀態標示為：

```text
preprocess_reference_pending_confirmation
```

意思是：

```text
這些值是 DataPreprocess 端的參考門檻，
可先用於資料對接與展示，
但尚未宣告為最終正式 rule output。
```


### 4-1. Threshold Reference 相關連檔案與目的

#### `docs/contracts/data_preprocess_threshold_reference.csv`

目的 / 主旨：

```text
以 CSV 表格整理 DataPreprocess 端提供的門檻參考值。
方便用 Excel 檢視，也方便後續轉入 sensor_threshold 或 rule config。
```

主要欄位：

```text
sensor_name
station
normal_min
normal_max
warning_min
warning_max
alarm_min
alarm_max
unit
threshold_source
status
note
```

---

#### `csv_templates/sensor_threshold_template.csv`

目的 / 主旨：

```text
以 sensor_threshold 的格式呈現門檻參考值。
可作為未來匯入 DB sensor_threshold 的欄位模板。
```

---

#### `templates/sensor_threshold.template.json`

目的 / 主旨：

```text
以 JSON template 呈現 sensor_threshold 參考資料格式。
目前不是空 placeholder，而是放入 DataPreprocess reference rows。
```

---

#### `schema/sensor_threshold.schema.json`

目的 / 主旨：

```text
定義 sensor_threshold 應有欄位。
包含 normal / warning / alarm 區間、unit、threshold_source、status、note 等欄位。
```

---

#### `ontology/knowledge/SprayLine_threshold_reference_knowledge.ttl`

目的 / 主旨：

```text
讓門檻參考值可以在 ontology / OntoGraf 中呈現。
用於展示 SensorSignal → SensorThresholdReference → EventRule 的關聯。
```

---

#### `ontology/event_rule/SprayLine_event_rule_ontology.ttl`

目的 / 主旨：

```text
描述 EventRule 與 threshold reference 的關係。
例如 FilterPressureEventRule 會 usesThresholdReference 到 TH_FILTER_DIFF_ALL。
```

---

#### `ontology/database/SprayLine_database_ontology.ttl`

目的 / 主旨：

```text
描述 sensor_threshold table 與 DB v3 的資料結構關係。
讓 threshold reference 可以與資料庫概念對應。
```


## 5. OntoGraf / Ontology 檢查與展示路徑

`0609ver_3` 已檢查 ontology 底下所有 `.ttl` 檔案的語法與基本關聯。

檢查報告：

```text
docs/validation/0609ver_3_ontology檢查報告.md
```

檢查內容包含：

```text
TTL parse
classes 數量
named individuals 數量
subClassOf 數量
object properties 數量
object assertions 數量
```

目前建議展示的 OntoGraf 路徑：

```text
SensorSignal
→ SensorThresholdReference
→ EventRule
→ alert_event
```

```text
FaultSymptom
→ FaultCause
→ CandidateResponse
```

```text
RESTEndpoint
→ ServiceFunction
→ DatabaseTable / ResponseSchema
```

```text
DatabaseZone
→ DatabaseTable / DatabaseView
```

```text
DashboardPanel
→ RESTEndpoint / DatabaseTable
```

補充：

```text
有些 class 是分類節點，打開後沒有下一層是正常情況。
若要看完整關聯，建議同時載入 ontology/ 裡所有 TTL，或先從 SprayLine_ontology_index.ttl 開始看。
```


## 6. 建議開檔案說明順序

進度說明時建議照這個順序打開檔案，不要一次開太多。

---

### 第 1 個：先開總說明

```text
README.md
```

說明：

```text
這版是 0609ver_3，主要整理 DB v3、DataPreprocess、Service、API、Ontology 拆檔，以及 threshold reference。
```

---

### 第 2 個：開 API 路由表

```text
docs/contracts/api_routes_0609.csv
```

說明：

```text
這裡整理所有 API。
可以先指出原因對策 API：
GET /troubleshooting/issues
GET /troubleshooting/issues/{issue_type}
```

也可以順便指出：

```text
POST /rules/evaluate
GET /alerts/unacknowledged
```

讓對方看到 rule 和 alert 沒有被拿掉。

---

### 第 3 個：開原因對策 Service Contract

```text
webservices/troubleshooting_service_contract/troubleshooting_service_contract.py
```

說明：

```text
這裡定義原因對策 service 未來要提供哪些 function。
目前是 contract，不是正式後端。
```

---

### 第 4 個：開原因對策候選資料

```text
knowledge/troubleshooting_matrix_candidate.csv
```

說明：

```text
這是原因對策候選知識來源。
目前 status 先標成 candidate_pending_confirmation。
```

---

### 第 5 個：開原因對策 schema

```text
schema/troubleshooting_matrix.schema.json
schema/troubleshooting_issue.schema.json
```

說明：

```text
這兩份定義原因對策 API / service 回傳格式。
```

---

### 第 6 個：開 DataPreprocess 門檻參考表

```text
docs/contracts/data_preprocess_threshold_reference.csv
```

說明：

```text
這裡整理 DataPreprocess 端提供的門檻參考值。
目前標示為 preprocess_reference_pending_confirmation。
```

---

### 第 7 個：開 sensor threshold template

```text
csv_templates/sensor_threshold_template.csv
templates/sensor_threshold.template.json
```

說明：

```text
這裡用 sensor_threshold 格式呈現門檻參考值。
未來可作為匯入 DB 或 rule config 的基礎。
```

---

### 第 8 個：開 ontology index

```text
ontology/SprayLine_ontology_index.ttl
```

說明：

```text
這是 ontology 總索引，說明 ontology 已拆成 core、database、service、dashboard、event_rule、troubleshooting、knowledge。
```

---

### 第 9 個：開 threshold reference ontology

```text
ontology/knowledge/SprayLine_threshold_reference_knowledge.ttl
```

說明：

```text
這裡放 DataPreprocess threshold reference 的 ontology graph 版本。
可以展示 SensorSignal → SensorThresholdReference → EventRule。
```

---

### 第 10 個：開 event rule ontology

```text
ontology/event_rule/SprayLine_event_rule_ontology.ttl
```

說明：

```text
Rule 沒有拿掉。
這裡描述異常判斷、event trigger、alert_event，以及 EventRule 如何參考 threshold reference。
```

---

### 第 11 個：開 troubleshooting ontology

```text
ontology/troubleshooting/SprayLine_troubleshooting_ontology.ttl
```

說明：

```text
這裡定義原因對策的概念，例如 FaultSymptom、FaultCause、CandidateResponse。
```

---

### 第 12 個：開 knowledge ontology

```text
ontology/knowledge/SprayLine_troubleshooting_knowledge.ttl
```

說明：

```text
這裡放具體候選原因對策知識，對應 knowledge/troubleshooting_matrix_candidate.csv。
```

---

### 第 13 個：開 service ontology

```text
ontology/service/SprayLine_service_ontology.ttl
```

說明：

```text
這裡描述 service、API endpoint、response schema 之間的關係。
TroubleshootingService 與原因對策 API 會在這裡被描述。
```

---

### 第 14 個：開 ontology 檢查報告

```text
docs/validation/0609ver_3_ontology檢查報告.md
```

說明：

```text
這裡記錄每個 ontology 檔案的 parse、class、individual、property、assertion 檢查結果。
```


## 7. 可以直接使用的口頭報告文字

這版我把前面提到的原因對策、service、API 和 ontology 拆開整理，並且在 `0609ver_3` 補上 DataPreprocess 端提供的門檻參考值。

首先，原因對策沒有直接寫成正式 rule，而是放在 `knowledge/troubleshooting_matrix_candidate.csv`，先作為候選知識。

再來，我有建立 `TroubleshootingService` 的 contract，放在 `webservices/troubleshooting_service_contract/troubleshooting_service_contract.py`，主要負責查原因對策，例如查某個 `issue_type` 可能對應的原因、影響設備、觀測訊號和候選處置。

API 的部分整理在 `docs/contracts/api_routes_0609.csv`，原因對策對應兩條 API：

```text
GET /troubleshooting/issues
GET /troubleshooting/issues/{issue_type}
```

Rule 的部分沒有拿掉，而是由 `EventRuleService` 負責判斷異常與觸發事件，例如：

```text
POST /rules/evaluate
GET /alerts/unacknowledged
```

`0609ver_3` 也把 DataPreprocess 端的門檻參考值整理到 `docs/contracts/data_preprocess_threshold_reference.csv`，並同步放到 `sensor_threshold` template 和 threshold reference ontology。

Ontology 的部分沒有全部放在同一個 TTL，而是拆成：

```text
core
database
service
dashboard
event_rule
troubleshooting
knowledge
```

其中 `event_rule` 負責描述異常觸發規則，`troubleshooting` 負責定義原因對策概念，`knowledge` 則放具體候選原因對策與 threshold reference。

所以現在的流程是：

```text
sensor data 進來後，
EventRuleService 會參考 sensor_threshold reference 判斷是否產生 alert_event；
如果產生事件，
再透過 TroubleshootingService 查原因與候選對策，
最後由 Dashboard 顯示給使用者。
```

這版不是把 rule 拿掉，而是把 rule 和原因對策分工清楚：

```text
rule 負責判斷異常，
TroubleshootingService 負責查可能原因與候選處置，
ontology 負責描述這些概念與關係。
```
